---

# Data Preprocessing

> This notebook contains code for preprocessing the corpus for training.
> This includes splitting latitudes, longitudes and normalization.
> The processed data is saved to `mappings.csv`

---

In [1]:
import os
import json
import numpy as np
import pandas as pd
import librosa
from tqdm.notebook import tqdm

CACHE_DIR = os.path.join('..', '.cache')
os.makedirs(CACHE_DIR, exist_ok=True)

## Load mappings.json

In [2]:
with open(os.path.join(CACHE_DIR, 'data.json'), 'r') as f:
    data = json.load(f)

In [3]:
paths = list(data.keys())
geotags = list(data.values())

**Get full file paths**

In [4]:
fullpaths = list()

for path in paths:
    fullpath = os.path.join(CACHE_DIR, 'audio', path)

    fullpaths.append(fullpath)

## Get latitudes and longitudes
- Split latitudes and longitudes
- Normalize them

In [5]:
lats = list()
longs = list()

for geotag in geotags:
    lat, long = geotag.split()
    lat = float(lat) / 90.0
    long = float(long) / 180.0

    lats.append(lat)
    longs.append(long)

## Write to csv
- Create mappings dataframe with full file paths, latitudes and longitudes
- Write to mappings.csv

In [6]:
df = pd.DataFrame({
    'path': fullpaths,
    'lat': lats,
    'long': longs
})

In [7]:
df.head()

,path,lat,long
0,../.cache/audio/33716.mp3,0.372981,-0.566150
1,../.cache/audio/33531.mp3,0.457793,0.051874
2,../.cache/audio/33530.mp3,0.457793,0.051874
3,../.cache/audio/33529.mp3,0.457793,0.051874
4,../.cache/audio/33521.mp3,0.583121,-0.011643


In [8]:
df.to_csv(os.path.join(CACHE_DIR, 'mappings.csv'), index=False)

## Process audio

In [19]:
def compress(audio, threshold=0.2, ratio=4):
    compressed = np.copy(audio)
    mask = np.abs(compressed) > threshold
    compressed[mask] = np.sign(compressed[mask]) * (
        threshold + (np.abs(compressed[mask]) - threshold) / ratio
    )
    return compressed


def normalize(audio):
    peak = np.max(np.abs(audio))
    if peak > 0:
        return audio / peak
    return audio


audio = list()

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing audio"):
    y, sr = librosa.load(row['path'])

    if y.shape[0] < 44100:
        y = np.pad(y, (0, max(0, 44100 - len(y))), mode="constant")

    # Pick the most harmonic part
    else:
        y = compress(y)
        harmonic, _ = librosa.effects.hpss(y)
        rms = librosa.feature.rms(y=harmonic[:-44100])[0]
        max_harm = librosa.frames_to_time(np.argmax(rms), sr=sr)
        y = y[int(max_harm * sr): int(max_harm * sr) + 44100]

    y = normalize(y)
    audio.append(y)

y = np.array(audio)
y.shape
np.save(os.path.join(CACHE_DIR, 'audio.npy'), y)

Processing audio:   0%|          | 0/5000 [00:00<?, ?it/s]